# Session 8: Spatial statistics, niches and where to go next

So far we have asked *what* each cell is. Spatial data lets us ask *where* it is, and *next to what*: do T cells
reach the tumour, or stay at its edge? Which cell types form neighbourhoods together? Do those neighbourhoods match
what a pathologist sees? In this session you will:

1. connect neighbouring cells into a **graph**, as QuPath's Delaunay clustering does (Day 1, session 4);
2. test which cell types are next to each other more often than chance (**neighbourhood enrichment**), and at what
   distances (**co-occurrence**);
3. group cells into **niches** by what surrounds them, and compare the niches with the pathology annotations;
4. find **communities** the way the Crick's CellSurvey pipeline does (Delaunay graph + Louvain);
5. take the results **back to QuPath**;
6. think about how to **evaluate the new tools** that keep appearing.

We use the cell types from session 7 for 10x's cells across the whole crop. The tools are
[squidpy](https://squidpy.readthedocs.io) for spatial statistics, scikit-learn and networkx.

Cells marked **Exercise** are for you to complete. The completed notebook is in `solutions/`.

## Setup

The cell types come from the end of session 7 (`session7_cell_types.csv`). If that file is missing, the notebook
uses the reference Day 1 phenotypes in the workshop data instead, and says so. The analysis works the same either
way; only the names of the groups change.

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import spatialdata as sd
import squidpy as sq
from scipy.spatial import Delaunay
from shapely import affinity
from shapely.geometry import mapping
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

DATA = Path("../data")
sdata = sd.read_zarr(DATA / "xenium_rcc_crop.zarr")
PIXEL_SIZE = sdata.attrs["pixel_size_um"]  # µm per image pixel

CELL_TYPES = DATA / "session7_cell_types.csv"
if CELL_TYPES.exists():
    cell_types = pd.read_csv(CELL_TYPES, index_col="cell_id")["cell_type"]
    print(f"Using the cell types from session 7 ({CELL_TYPES.name})")
else:
    reference = gpd.read_file(DATA / "qupath" / "xenium_rcc_crop_10x_phenotypes.geojson")
    cell_types = pd.Series([c["name"] for c in reference["classification"]], index=reference["name"])
    print("session7_cell_types.csv not found: using the reference Day 1 phenotypes instead")

# One row per typed cell: its protein intensities, its position (obsm["spatial"], in µm) and its cell type
adata = sdata["protein"][sdata["protein"].obs_names.isin(cell_types.index)].copy()
adata.obs["cell_type"] = pd.Categorical(cell_types.reindex(adata.obs_names).values)
print(f"{adata.n_obs:,} cells")
adata.obs["cell_type"].value_counts()

A reminder of where the cell types are. Each dot is a cell's centre, which is quicker to draw than 14,000 outlines.
The pathologist's annotations from Day 1 are outlined in black.

In [ ]:
COLOURS = dict(zip(adata.obs["cell_type"].cat.categories, plt.cm.tab10.colors))
annotations = sd.transform(sdata["pathology_annotations"], to_coordinate_system="microns")
xy = adata.obsm["spatial"]  # cell centres, in µm


def cell_map(labels, colours, ax, title, size=2):
    """Plot each cell's centre in its label's colour, with the pathology annotations on top."""
    for name, colour in colours.items():
        chosen = np.asarray(labels) == name
        ax.scatter(*xy[chosen].T, s=size, color=colour, label=f"{name} ({chosen.sum():,})", linewidths=0)
    annotations.boundary.plot(ax=ax, color="black", linewidth=1)
    ax.set(xlabel="x (µm)", ylabel="y (µm)", title=title, aspect="equal")
    ax.invert_yaxis()  # images have y pointing down
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize="small", markerscale=4)


fig, ax = plt.subplots(figsize=(11, 9))
cell_map(adata.obs["cell_type"], COLOURS, ax, "Cell types")
for _, row in annotations.iterrows():
    point = row.geometry.representative_point()
    ax.annotate(row["name"], (point.x, point.y), fontsize="large", ha="center")
plt.show()

## 1. Which cells are neighbours?

Every spatial statistic starts by deciding which cells count as neighbours. A **Delaunay triangulation** joins
each cell to its nearest surrounding cells with triangles, so that no cell lies inside another triangle's circle.
It's the same graph as QuPath's *Analyze › Spatial analysis › Delaunay clustering* from Day 1.

squidpy builds it with `spatial_neighbors_delaunay` and stores two cell-by-cell tables in `adata.obsp`: which cells are
connected (`spatial_connectivities`) and how far apart they are (`spatial_distances`, in µm here).

In [ ]:
sq.gr.spatial_neighbors_delaunay(adata)
lengths = adata.obsp["spatial_distances"].data  # one value per edge (each counted from both ends)
print(f"{len(lengths) // 2:,} edges; median length {np.median(lengths):.1f} µm, longest {lengths.max():.0f} µm")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=np.arange(0, 200, 2), log=True)
ax.set(xlabel="edge length (µm)", ylabel="edges")
plt.show()

Most edges join cells about 10 µm apart, but the triangulation joins *every* cell to something, so cells on either
side of an empty space (a vessel, a tear, the edge of the tissue) get joined by long edges that mean nothing.

**Exercise 1.** How many edges are longer than 20, 30, 50 and 100 µm? Draw the edges longer than 30 µm on the map
of cell types: where are they? (The pairs of cells for each edge are in `adata.obsp["spatial_distances"]`;
`.tocoo()` gives them as `row`, `col` and `data`.)

In [ ]:
# Your code here

We remove edges longer than a cut-off, as CellSurvey does and as QuPath's Delaunay clustering does with its
"distance threshold". The function below keeps only the edges up to `max_um` long. (squidpy can do this for you,
with `sq.gr.spatial_neighbors_delaunay(adata, radius=30)`; we do it by hand to see what happens.)

In [ ]:
MAX_EDGE_UM = 30


def remove_long_edges(adata, max_um):
    """Drop graph edges longer than max_um from both spatial_connectivities and spatial_distances."""
    distances = adata.obsp["spatial_distances"].tocsr()
    keep = distances.copy()
    keep.data = (keep.data <= max_um).astype(float)
    for key in ("spatial_connectivities", "spatial_distances"):
        matrix = adata.obsp[key].tocsr().multiply(keep).tocsr()
        matrix.eliminate_zeros()
        adata.obsp[key] = matrix


remove_long_edges(adata, MAX_EDGE_UM)
neighbours = np.diff(adata.obsp["spatial_connectivities"].indptr)  # neighbours per cell
print(f"{adata.obsp['spatial_connectivities'].nnz // 2:,} edges left; "
      f"median {np.median(neighbours):.0f} neighbours per cell; {np.sum(neighbours == 0)} cells with none")

## 2. Who is next to whom?

### Neighbourhood enrichment

Count, for every pair of cell types, how many graph edges join them. Is that more or less than you'd expect by
chance? squidpy answers by **permutation**: it shuffles the cell type labels many times, keeping every cell where it
is, and counts again. The result is a **z-score** per pair: positive means the two types are neighbours more often
than chance, negative less often. (200 permutations take a few seconds; more give steadier numbers.)

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="cell_type", n_perms=200, rng=0, show_progress_bar=False)
types = adata.obs["cell_type"].cat.categories
zscores = pd.DataFrame(adata.uns["cell_type_nhood_enrichment"]["zscore"], index=types, columns=types)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(zscores, cmap="RdBu_r", center=0, vmin=-40, vmax=40, annot=True, fmt=".0f", ax=ax,
            cbar_kws={"label": "z-score (above 40 shown as 40)"})
ax.set_title("Neighbourhood enrichment")
plt.show()

**Questions**

- The diagonal is strongly positive: cells of a type sit next to cells of the same type. Why might that be, beyond
  biology? (Think about how the cells were typed, and about segmentation.)
- Which pairs are enriched off the diagonal? Do they make biological sense (for example, blood vessels are lined
  by endothelium and wrapped in smooth muscle)?
- Are T cells enriched next to tumour cells, or depleted? What would each mean for a patient?
- The z-scores are huge. With 14,000 cells almost any difference is "significant". Is a large z-score the same as
  a large effect?

### Co-occurrence at different distances

Enrichment only looks at direct neighbours. **Co-occurrence** asks the same question at a range of distances: how
much more likely is cell type B within distance *d* of a cell of type A, than anywhere? A value of 1 means "as
often as chance".

In [ ]:
sq.gr.co_occurrence(adata, cluster_key="cell_type", interval=np.linspace(10, 200, 20))
co_occurrence = adata.uns["cell_type_co_occurrence"]
distances = co_occurrence["interval"][1:]


def plot_co_occurrence(cell_type, ax):
    """How much more likely each cell type is within each distance of `cell_type`, than anywhere."""
    i = list(types).index(cell_type)
    for j, other in enumerate(types):
        ax.plot(distances, co_occurrence["occ"][i, j], label=other, color=COLOURS[other])
    ax.axhline(1, color="grey", linestyle="--")
    ax.set(xlabel="distance (µm)", ylabel="ratio to chance", title=f"Near {cell_type} cells", yscale="log")


fig, ax = plt.subplots(figsize=(8, 5))
plot_co_occurrence("Tumour" if "Tumour" in types else types[0], ax)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize="small")
plt.show()

**Exercise 2.** Plot the co-occurrence for two other cell types side by side, for example the T cells and the
macrophages (`types` lists the names). Which cell types are found near them, and does that change with distance?

In [ ]:
# Your code here

## 3. Niches: cells grouped by their surroundings

A **niche** (or "cellular neighbourhood") is a kind of surrounding that recurs across the tissue: for example,
"tumour cells mixed with macrophages", or "a B-cell follicle". One widely used way to find them:

1. for each cell, list the cell types within a radius (say 50 µm) and turn the counts into fractions;
2. cluster the cells by those fractions, with k-means.

Cells in the same niche have similar surroundings, whatever their own type.

In [ ]:
def neighbourhood_composition(radius_um):
    """For each cell, the fraction of each cell type among the cells within radius_um (including itself)."""
    nearby = NearestNeighbors(radius=radius_um).fit(xy).radius_neighbors(xy, return_distance=False)
    one_hot = pd.get_dummies(adata.obs["cell_type"]).to_numpy(float)
    return pd.DataFrame([one_hot[i].mean(axis=0) for i in nearby], columns=types, index=adata.obs_names)


def find_niches(composition, n_niches):
    """Cluster cells by their neighbourhood composition. Returns niche labels (1, 2, ...)."""
    kmeans = KMeans(n_niches, n_init=10, random_state=0).fit(composition)
    return pd.Series(kmeans.labels_ + 1, index=composition.index).astype(str).radd("Niche ")


RADIUS_UM, N_NICHES = 50, 6
composition = neighbourhood_composition(RADIUS_UM)
adata.obs["niche"] = find_niches(composition, N_NICHES)

# What each niche contains: the mean composition of the neighbourhoods in it
profile = composition.groupby(adata.obs["niche"]).mean()
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(profile, cmap="viridis", annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "mean fraction"})
ax.set_title(f"What surrounds the cells in each niche (within {RADIUS_UM} µm)")
plt.show()

In [ ]:
NICHE_COLOURS = dict(zip(sorted(adata.obs["niche"].unique()), plt.cm.Set2.colors))
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
cell_map(adata.obs["cell_type"], COLOURS, axes[0], "Cell types")
cell_map(adata.obs["niche"], NICHE_COLOURS, axes[1], f"Niches ({RADIUS_UM} µm, k = {N_NICHES})")
plt.tight_layout()
plt.show()

**Do the niches match the pathology annotations?** Each cell gets the annotation its centre falls in (or "Not
annotated"). The table shows how each annotated region is made up of niches.

In [ ]:
centres = gpd.GeoDataFrame(geometry=gpd.points_from_xy(*xy.T), index=adata.obs_names)
in_annotation = centres.sjoin(annotations[["name", "geometry"]], how="left", predicate="within")
adata.obs["annotation"] = in_annotation[~in_annotation.index.duplicated()]["name"].fillna("Not annotated")


def compare_with_annotations(labels, title):
    table = pd.crosstab(labels, adata.obs["annotation"], normalize="columns").mul(100)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(table, cmap="Blues", annot=True, fmt=".0f", ax=ax, cbar_kws={"label": "% of the region's cells"})
    ax.set(xlabel="pathology annotation", ylabel="", title=title)
    plt.show()
    print(f"Agreement with the annotations (adjusted Rand index): {adjusted_rand_score(adata.obs['annotation'], labels):.2f}")


compare_with_annotations(adata.obs["niche"], "Niches in each annotated region")

**Exercise 3.** The radius and the number of niches are your choices, like the threshold in session 5. Try a smaller
and a larger radius (for example 25 and 100 µm) and a different number of niches (for example 4 and 10). Which
niches stay, and which split or merge? Which settings agree best with the annotations?

In [ ]:
# Your code here

**Questions**

- The annotations have three categories and your niches several. Is low agreement a failure of the niches, or of
  the annotations? What can niches show that a pathologist's outline can't, and the other way round?
- Niches depend on the cell types, which depend on the clustering (session 7), which depends on the segmentation
  (sessions 5 and 6). Where in that chain would you check first, if a niche looked odd?

## 4. Communities, the CellSurvey way

The Crick's [CellSurvey](https://github.com/FrancisCrickInstitute/CellSurvey) pipeline takes a different route to
the same question. It builds the Delaunay graph, removes edges longer than a cut-off, and gives each edge a
**weight**: 1 plus the correlation between the two cells' marker intensities, so similar cells are joined more
strongly. **Louvain community detection** then splits the graph into communities: groups of cells more connected to
each other than to the rest. Unlike niches, communities are always connected patches of tissue.

The `resolution` sets the scale: smaller values give fewer, larger communities.

In [ ]:
def louvain_communities(max_edge_um=50, resolution=0.1, seed=42):
    """CellSurvey's communities: a Delaunay graph weighted by marker similarity, split with Louvain."""
    triangles = Delaunay(xy).simplices
    pairs = np.vstack([triangles[:, [0, 1]], triangles[:, [1, 2]], triangles[:, [0, 2]]])
    pairs = np.unique(np.sort(pairs, axis=1), axis=0)  # each edge once
    pairs = pairs[np.linalg.norm(xy[pairs[:, 0]] - xy[pairs[:, 1]], axis=1) <= max_edge_um]

    intensities = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    markers = np.arcsinh(intensities / 5)  # as in session 7
    markers -= markers.mean(axis=1, keepdims=True)
    markers /= np.linalg.norm(markers, axis=1, keepdims=True) + 1e-9
    correlation = np.sum(markers[pairs[:, 0]] * markers[pairs[:, 1]], axis=1)  # per edge

    graph = nx.Graph()
    graph.add_nodes_from(range(len(xy)))
    graph.add_weighted_edges_from(zip(pairs[:, 0].tolist(), pairs[:, 1].tolist(), (1 + correlation).tolist()))
    communities = nx.community.louvain_communities(graph, weight="weight", resolution=resolution, seed=seed)

    labels = np.empty(len(xy), dtype=object)
    for number, members in enumerate(sorted(communities, key=len, reverse=True), start=1):
        labels[list(members)] = f"Community {number}"
    return pd.Series(labels, index=adata.obs_names)


for resolution in (0.1, 0.01):
    adata.obs[f"community_{resolution}"] = louvain_communities(resolution=resolution)
    sizes = adata.obs[f"community_{resolution}"].value_counts()
    print(f"resolution {resolution}: {len(sizes)} communities; largest {sizes.head(5).tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, resolution in zip(axes, (0.1, 0.01)):
    labels = adata.obs[f"community_{resolution}"]
    names = labels.value_counts().index
    colours = dict(zip(names, plt.cm.tab10.colors if len(names) <= 10 else plt.cm.tab20.colors))
    cell_map(labels, colours, ax, f"Louvain communities, resolution {resolution}")
plt.tight_layout()
plt.show()

for resolution in (0.1, 0.01):
    compare_with_annotations(adata.obs[f"community_{resolution}"], f"Communities (resolution {resolution})")

**Questions**

- Which agrees better with the pathologist: niches or communities? At which resolution? Why might large, connected
  communities match hand-drawn outlines better than niches do?
- At resolution 0.1, the tumour region is split into large patches. Look at the cell type map: is there a real
  boundary between them? What does that tell you about reading biology into community borders?
- A niche can appear in many places; a community is one place. Which would you use to ask "is this kind of
  neighbourhood more common in responders than in non-responders"?
- (Very small communities, of a cell or two, are isolated cells left after removing long edges.)

## 5. Back to QuPath

Numbers are easier to trust once you've seen them on the image. The cell below saves 10x's cells as QuPath
detections: each is classified by its cell type, and carries its niche and community as **measurements**, which
QuPath can colour cells by.

In QuPath, open `xenium_rcc_crop_if.ome.tif`, then **File › Import objects from file** and choose
`session8_cells.geojson`. Then use **Measure › Show measurement maps** and pick `Niche` or `Community`. Compare them
with the cells you classified on Day 1.

In [ ]:
outlines = sdata["cell_boundaries"].loc[adata.obs_names]  # 10x's outlines, in µm
niche_numbers = adata.obs["niche"].str.removeprefix("Niche ").astype(int)
community_numbers = adata.obs["community_0.1"].str.removeprefix("Community ").astype(int)

features = []
for cell_id, geometry in outlines.geometry.items():
    cell_type = adata.obs.at[cell_id, "cell_type"]
    in_pixels = affinity.scale(geometry, 1 / PIXEL_SIZE, 1 / PIXEL_SIZE, origin=(0, 0))  # QuPath uses pixels
    features.append({
        "type": "Feature",
        "geometry": mapping(in_pixels),
        "properties": {
            "objectType": "detection",
            "name": cell_id,  # the 10x cell ID, as in the Day 1 files
            "classification": {"name": cell_type, "color": [round(255 * c) for c in COLOURS[cell_type][:3]]},
            "measurements": {"Niche": int(niche_numbers[cell_id]), "Community": int(community_numbers[cell_id])},
        },
    })

OUTPUT = DATA / "session8_cells.geojson"
OUTPUT.write_text(json.dumps({"type": "FeatureCollection", "features": features}))
print(f"Saved {len(features):,} cells to {OUTPUT}")

## 6. Evaluating new tools

New methods for spatial data appear every week: segmenters, cell typers, niche finders, foundation models. You
can't try them all, and you shouldn't trust any of them blindly. Before investing time in one, ask:

1. **What question does it answer**, and is it yours? (Segmentation, cell typing, niches, comparing samples?)
2. **What does it need as input?** Raw images, transcripts, a cell table, cell types, marker lists? Which
   technologies was it built and tested on?
3. **How was it validated?** On data like yours? Against what ground truth, and compared with which simpler
   methods? Is there an independent benchmark?
4. **Can you run it?** Language (Python or R), installation, GPU, memory and time on your data size. Does it read
   standard formats (SpatialData, AnnData, OME-TIFF, GeoJSON) or need converters?
5. **Is it maintained?** Recent releases, open issues answered, documentation and tutorials, a licence that allows
   your use.
6. **What are its parameters**, and how sensitive are the results to them? (Every session today had a threshold,
   a radius, a resolution or a number of clusters that changed the answer.)
7. **Can you check the output?** Can you get it back onto the image, in QuPath or napari, and see whether it makes
   sense?

Some tools for the steps we covered, from the survey behind CellSurvey (a starting point, not a recommendation):

| Step | Tools to look at |
|---|---|
| Segmentation | [InstanSeg](https://github.com/instanseg/instanseg) and Cellpose (images); [segger](https://github.com/dpeerlab/segger), Baysor and Proseg (transcripts), several of them run through [Sopa](https://prism-oncology.github.io/sopa) |
| Segmentation QC | [SegTraQ](https://github.com/LazDaria/segtraq) |
| Cell typing | scanpy clustering; [CELESTA](https://github.com/plevritis-lab/CELESTA) (marker-based, with a spatial prior; R) |
| Niches and domains | [Novae](https://github.com/prism-oncology/novae) (a graph foundation model); [SACCELERATOR](https://github.com/SpatialHackathon/SACCELERATOR) (benchmarks spatial clustering methods) |
| Comparing samples | [PANORAMIC](https://github.com/plevritis-lab/panoramic) (co-localisation across patients and groups; R) |
| Registration | [VALIS](https://github.com/MathOnco/VALIS) (aligning serial sections and imaging cycles) |
| H&E and pathology models | [LazySlide](https://github.com/rendeirolab/LazySlide) |
| Viewing | QuPath; [napari-spatialdata](https://github.com/scverse/napari-spatialdata) |
| Whole pipeline | [CellSurvey](https://github.com/FrancisCrickInstitute/CellSurvey) (the notebooks' steps, from image to communities) |

**Discussion.** Pick one tool from the table (or one you've read about) and go through the seven questions for your
own project. What would you need to find out before using it?

## Summary

- Spatial statistics start with a graph of neighbours, and the graph needs choices too: Delaunay or a radius, and a
  cut-off for long edges.
- Neighbourhood enrichment and co-occurrence test which cell types sit together, against a shuffled null. With many
  cells, look at the size of an effect, not just its significance.
- Niches group cells by their surroundings; communities split the tissue into connected patches. Both depend on a
  scale you choose, and on every step before them.
- Results belong back on the image: QuPath reads the same GeoJSON, so you can check them by eye.
- Judge new tools by the question they answer, how they were validated, whether you can run and check them, and
  how sensitive they are to their settings.

**Further reading**

- [squidpy tutorials](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/index.html).
- Schürch et al. (2020), [Coordinated cellular neighborhoods orchestrate antitumoral immunity at the colorectal cancer
  invasive front](https://doi.org/10.1016/j.cell.2020.07.005), *Cell*: the cellular-neighbourhood approach of section 3.
- Blondel et al. (2008), [Fast unfolding of communities in large networks](https://doi.org/10.1088/1742-5468/2008/10/P10008):
  the Louvain method.